# Institutional Portfolio & Risk Analytics Dashboard

# Notebook 1: 
Download Historical Stock Data

# Objective: 
Download historical stock price data using Python and prepare it for portfolio analysis.

# Author:
Ekta Lanjewar

# workflow

Import libraries
        ↓
Choose portfolio
        ↓
Download data
        ↓
Check currency and history
        ↓
Inspect dataset
        ↓
Check missing values
        ↓
Extract Adj Close
        ↓
Handle missing values
        ↓
Verify data quality
        ↓
Download and align the benchmark
        ↓
Save data

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Nine eurozone companies, all listed and traded in EUR
tickers = [
    "ASML.AS",   # ASML Holding - Euronext Amsterdam
    "BNP.PA",    # BNP Paribas - Euronext Paris
    "MC.PA",     # LVMH - Euronext Paris
    "SIE.DE",    # Siemens - Xetra (Frankfurt)
    "AIR.PA",    # Airbus - Euronext Paris
    "OR.PA",     # L'Oréal - Euronext Paris
    "SAN.PA",    # Sanofi - Euronext Paris
    "SAP.DE",    # SAP - Xetra (Frankfurt)
    "ALV.DE"     # Allianz - Xetra (Frankfurt)
]

# Benchmark: Xtrackers EURO STOXX 50 UCITS ETF 1C (Xetra, EUR).
# It is an accumulating fund (dividends are reinvested) that tracks the
# EURO STOXX 50 Net Return index. All nine holdings belong to this index.
benchmark = "XESC.DE"

start_date = "2019-01-01"
end_date = "2025-12-31"
data = yf.download(tickers, start=start_date, end=end_date, auto_adjust=False)

data.head()

## Check that every ticker is traded in EUR

The portfolio is analysed from the perspective of an **unhedged euro-based investor holding euro-denominated shares**. Because every holding and the benchmark are priced in EUR, no currency conversion is needed and returns contain no exchange-rate effect.

We confirm the trading currency that Yahoo Finance reports for each ticker. The notebook stops if any ticker is not in EUR.

In [ ]:
for ticker in tickers + [benchmark]:
    currency = yf.Ticker(ticker).fast_info["currency"]
    print(f"{ticker:8} {currency}")
    assert currency == "EUR", f"{ticker} is not traded in EUR"

# Check object type
type(data)

# Check dataset dimensions
data.shape

# View column names
data.columns

# View row index (dates)
data.index

# Summarize dataset
data.info()

# Count missing values
data.isna().sum()

In [ ]:
data.shape

1. The number of rows is the number of trading days in the sample (one row per date on which at least one of the exchanges was open).
2. For each stock, Yahoo Finance returned multiple fields:
Open
High
Low
Close
Adj Close
Volume

Since you downloaded 9 stocks, there are:

6 fields × 9 stocks = 54 columns

In [ ]:
data.columns

In [ ]:
type(data)

In [ ]:
data.index #This tells us the index contains dates and times. "Every row in this DataFrame is identified by a date.

## Data Inspection (data.info)

In this section, we inspect the downloaded dataset to understand:

- its structure,
- data types,
- missing values (NaN),
- and overall quality before performing any analysis.

In [ ]:
data.info()

## Missing Values

The portfolio contains stocks listed on Euronext Paris, Euronext Amsterdam and Xetra (Frankfurt). These exchanges share most holidays, but not all of them (for example, Xetra is closed on 24 and 31 December, when Euronext is open), so a few dates contain missing observations. We inspect these missing values before deciding on a cleaning strategy.

# Check for missing (NaN) values in each column.
# isna() identifies missing values, and sum() counts them.
# This helps assess data quality before analysis.

In [ ]:
data.isna().sum()

## Select Adjusted Closing Prices

Portfolio returns should be calculated using Adjusted Closing Prices because they account for stock splits and dividend adjustments.

In [ ]:
adj_close= data['Adj Close']
adj_close

## Check the historical coverage of each stock

Each stock must have prices for the whole period from January 2019 to December 2025. The table shows the first and last available price and the number of missing values before cleaning.

In [ ]:
coverage = pd.DataFrame({
    "first_price": adj_close.apply(lambda s: s.first_valid_index().date()),
    "last_price": adj_close.apply(lambda s: s.last_valid_index().date()),
    "missing_values": adj_close.isna().sum()
})
print(coverage)

# Every stock must start at the beginning and end at the end of the sample
assert (adj_close.apply(lambda s: s.first_valid_index()) == adj_close.index[0]).all(), "A stock starts late"
assert (adj_close.apply(lambda s: s.last_valid_index()) == adj_close.index[-1]).all(), "A stock ends early"

## Handle Missing Values

The portfolio contains stocks listed on different exchanges, which observe different trading holidays. As a result, some dates contain missing (`NaN`) prices.

Using `dropna()` would remove every row containing at least one missing value, causing unnecessary loss of valid market data.

Instead, `ffill()` (forward fill) replaces each missing value with the most recent available price, preserving the time series for portfolio analysis.

### Example

Before:

| Date | Siemens |
|------|---------|
| 23 Dec | 230 |
| 24 Dec | NaN |
| 25 Dec | NaN |
| 26 Dec | 235 |

After `ffill()`:

| Date | Siemens |
|------|---------|
| 23 Dec | 230 |
| 24 Dec | 230 |
| 25 Dec | 230 |
| 26 Dec | 235 |

In [ ]:
adj_close= adj_close.ffill()
adj_close.head() #Printing all 1,809 rows clutters the notebook. Usually we inspect only the beginning or end of the dataset.

In [ ]:
adj_close.isna().sum() 

## Check the adjusted prices for unusual moves

Adjusted closing prices should reflect dividends, stock splits and spin-offs. A data error usually appears as an unusually large one-day move. The largest daily moves of each stock are listed so they can be checked against known market events.

Known events in this period include:

- **March 2020**: COVID-19 market crash (large moves in every stock).
- **28 September 2020 (SIE.DE)**: spin-off of Siemens Energy, one Siemens Energy share for every two Siemens shares. In the previously downloaded data, Siemens showed a gain of about +9% on this day. This is consistent with reports that the Siemens share price fell only slightly while shareholders also received the Siemens Energy shares, so the adjusted price correctly reflects the spin-off.
- **6 May 2022 (SAN.PA)**: spin-off of EUROAPI, one EUROAPI share for every 23 Sanofi shares (less than 1% of Sanofi's value).
- **13 November 2025 (SIE.DE)**: Siemens shares fell almost 10% after its annual results.

Any other large move should be investigated before the data is used.

In [ ]:
daily_returns = adj_close.pct_change().dropna()

for ticker in daily_returns.columns:
    largest_moves = daily_returns[ticker].abs().nlargest(3)
    moves = ", ".join(f"{date.date()} ({daily_returns.loc[date, ticker]:+.1%})" for date in largest_moves.index)
    print(f"{ticker:8} {moves}")

## Download and align the benchmark

The benchmark is downloaded for the same period and aligned with the portfolio's dates:

- `reindex(adj_close.index)` keeps exactly the portfolio's dates, so the portfolio and the benchmark cover the same period.
- `ffill()` fills a date with no benchmark price (for example a day when Xetra was closed but Euronext was open) with the most recent previous price, the same treatment as the stock prices.
- Values are never filled backwards (`bfill`), because that would use future information (look-ahead bias).

We use the **Adjusted Close**, as for the stocks. Because the fund is accumulating, it does not pay dividends, so its Adjusted Close and Close should be almost identical.

In [ ]:
benchmark_data = yf.download(benchmark, start=start_date, end=end_date, auto_adjust=False)

benchmark_prices = benchmark_data["Adj Close"]
# yfinance may return a one-column table instead of a Series
if isinstance(benchmark_prices, pd.DataFrame):
    benchmark_prices = benchmark_prices.iloc[:, 0]
benchmark_prices = benchmark_prices.dropna()

print("Benchmark data from", benchmark_prices.index.min().date(), "to", benchmark_prices.index.max().date())

# The benchmark must cover the whole portfolio period
assert benchmark_prices.index.min() <= adj_close.index.min(), "Benchmark starts after the portfolio"
assert benchmark_prices.index.max() >= adj_close.index.max(), "Benchmark ends before the portfolio"

In [ ]:
# Keep exactly the portfolio's dates, then fill gaps with the previous price only
benchmark_aligned = benchmark_prices.reindex(adj_close.index)
print("Portfolio dates with no benchmark price (filled with the previous day):", benchmark_aligned.isna().sum())

benchmark_aligned = benchmark_aligned.ffill()
benchmark_aligned.name = benchmark

assert benchmark_aligned.notna().all(), "Benchmark price missing on the first portfolio date"
assert benchmark_aligned.index.equals(adj_close.index)

# how to save as csv
1. ..      = Go back one folder
2. /data   = Enter the data folder
3. adj_close.csv = Save the file with this name

In [ ]:
import os

# Create the data folder if it does not exist yet (for example in a fresh copy of the project)
os.makedirs("../data", exist_ok=True)

# Stock prices in EUR (used by Notebooks 2 and 3 and the dashboard)
adj_close.to_csv("../data/adj_close.csv")

# Benchmark prices in EUR, on the same dates as the portfolio
benchmark_aligned.to_csv("../data/benchmark.csv")

In [ ]:
import os

os.listdir("../data")